In [3]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
import joblib

# File Paths
X_CSV = "../data stuff/augmented_telemetry_filtered_with_location.csv"
Y_CSV = "../data stuff/next24h_consumption.csv"
OUT_CSV = "../data stuff/predicted_24h_consumption_per_location.csv"
FINAL_REPORT_CSV = "../data stuff/final_comparison_report.csv"

def run_demand_prediction():
    df_x = pd.read_csv(X_CSV)
    df_y = pd.read_csv(Y_CSV)

    # 1. Prepare Daily History
    df_x['recorded_at'] = pd.to_datetime(df_x['recorded_at'])
    df_x['day_of_week'] = df_x['recorded_at'].dt.dayofweek
    
    # Aggregate into daily totals
    train_agg = df_x.groupby(['location_id', 'date']).agg({
        'dispensed_l': 'sum',
        'pressure_pa': 'mean',
        'device_id': 'nunique',
        'day_of_week': 'first'
    }).reset_index()

    # 2. Add LAG FEATURE
    train_agg = train_agg.sort_values(['location_id', 'date'])
    train_agg['prev_day_l'] = train_agg.groupby('location_id')['dispensed_l'].shift(1)
    train_agg['prev_day_l'] = train_agg['prev_day_l'].fillna(train_agg.groupby('location_id')['dispensed_l'].transform('mean'))

    # 3. Train the Model
    features = ['prev_day_l', 'pressure_pa', 'device_id', 'day_of_week']
    X_train = train_agg[features]
    y_train = train_agg['dispensed_l']

    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    joblib.dump(model, 'water_demand_model.pkl')
    joblib.dump(features, 'model_features.pkl')
    print("Model assets saved: 'water_demand_model.pkl' and 'model_features.pkl'")

    # 4. Prepare Test Data
    dev_counts = df_x.groupby('location_id')['device_id'].nunique().to_dict()
    last_day_consumption = train_agg.sort_values('date').groupby('location_id').last()['dispensed_l'].to_dict()
    
    latest_snapshots = df_x.sort_values('recorded_at').groupby('location_id').last().reset_index()
    latest_snapshots['device_id'] = latest_snapshots['location_id'].map(dev_counts)
    latest_snapshots['prev_day_l'] = latest_snapshots['location_id'].map(last_day_consumption)
    
    predict_df = latest_snapshots[latest_snapshots['location_id'].isin(df_y['location_id'])].copy()
    X_test = predict_df[features]
    
    # 5. Predict and Evaluate
    predict_df['model_forecast_l'] = model.predict(X_test)
    
    final_comparison = df_y.merge(
        predict_df[['location_id', 'model_forecast_l']], 
        on='location_id', 
        how='left'
    )

    # Output Metrics
    actual = final_comparison['predicted_demand_l']
    predicted = final_comparison['model_forecast_l']
    print(" MODEL PERFORMANCE")
    print(f"MSE: {round(mean_squared_error(actual, predicted), 2)}")
    print(f"MAE: {round(mean_absolute_error(actual, predicted), 2)} Liters")

    # Save intermediate predictions
    final_comparison.to_csv(OUT_CSV, index=False)
    return final_comparison

def generate_comparison_report(df_results):
    
    # Rename columns
    df = df_results.rename(columns={
        'predicted_demand_l': 'actual_avg_demand_l',
        'model_forecast_l': 'ml_predicted_demand_l'
    })

    # Calculate Variance and Accuracy
    df['variance_l'] = df['ml_predicted_demand_l'] - df['actual_avg_demand_l']
    df['accuracy_pct'] = (1 - (abs(df['variance_l']) / df['actual_avg_demand_l'])) * 100

    # Priority Logic
    q_high = df['ml_predicted_demand_l'].quantile(0.75)
    df['priority_level'] = df['ml_predicted_demand_l'].apply(
        lambda x: 'CRITICAL' if x > q_high else 'NORMAL'
    )

    # Rounding
    df = df.round({
        'actual_avg_demand_l': 1, 
        'ml_predicted_demand_l': 1, 
        'variance_l': 1, 
        'accuracy_pct': 1
    })

    # Save Final Report
    cols = ['location_id', 'actual_avg_demand_l', 'ml_predicted_demand_l', 'variance_l', 'accuracy_pct', 'priority_level']
    df[cols].to_csv(FINAL_REPORT_CSV, index=False)
    
    print(f"Final Comparison Report saved to: {FINAL_REPORT_CSV}")
    print("\nTop 5 Predictions vs Actuals")
    print(df[['location_id', 'actual_avg_demand_l', 'ml_predicted_demand_l', 'priority_level']].head())

if __name__ == "__main__":
    results = run_demand_prediction()
    generate_comparison_report(results)

Model assets saved: 'water_demand_model.pkl' and 'model_features.pkl'
 MODEL PERFORMANCE
MSE: 126634.61
MAE: 292.99 Liters
Final Comparison Report saved to: ../data stuff/final_comparison_report.csv

Top 5 Predictions vs Actuals
  location_id  actual_avg_demand_l  ml_predicted_demand_l priority_level
0     loc_001               2965.0                 2762.5         NORMAL
1     loc_002               2812.8                 2744.0         NORMAL
2     loc_003               6364.5                 6611.6       CRITICAL
3     loc_004               3104.1                 2897.8         NORMAL
4     loc_005               3617.9                 3516.0         NORMAL


In [4]:
def test_inference(prev_day, pressure, devices, day_idx):

    model = joblib.load('water_demand_model.pkl')
    features = joblib.load('model_features.pkl')
    
    data = {
        'prev_day_l': [prev_day],
        'pressure_pa': [pressure],
        'device_id': [devices],
        'day_of_week': [day_idx]
    }
    input_df = pd.DataFrame(data)
    
    prediction = model.predict(input_df)[0]
    return round(prediction, 2)

result = test_inference(3200, 4000.0, 5, 0)
print(f"Prediction for test case: {result} Liters")

Prediction for test case: 3131.47 Liters
